# Advanced RAG — Ingestion experiments (script-backed)

This notebook is for **reviewing and experimenting**, not a replacement for your `Ingestor` class. It adds checklist items **8, 10, 11, 18, 19**:

8. Structure-aware chunking  
10. Exact duplicate handling  
11. Near-duplicate detection  
18. Embedding reuse on updates  
19. End-to-end validation and recovery checks

**Import contract:** put your existing Python class in `ingestor.py` alongside this notebook. The notebook expects the corrected script discussed earlier, exporting `Ingestor`, `IngestorConfig`, `chunk_source`, `standardized_docs`, `source_id`, `source_key`, `file_hash`, `digest`, `point_id`, `delete_chunk_ids`, and `existing_manifest`. If your script has a different filename or class name, change only the import cell.

**Safety:** all example inspections are read-only. The optimized ingestion function is defined but **never called automatically**. It performs real writes and may incur Gemini API costs when you explicitly call it.

## 0 — Dependencies and imports

Use the same virtual environment and `.env` as your existing script. We reuse the class instead of copying loader/manifest/Qdrant setup into the notebook.

In [1]:
# Run once if needed:
# %pip install -U langchain-text-splitters rapidfuzz qdrant-client

from pathlib import Path
from collections import defaultdict
from rapidfuzz.fuzz import ratio
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter
from qdrant_client import models
import json

from ingestor import (
    Ingestor, IngestorConfig,
    standardized_docs, chunk_source,
    source_id, source_key, file_hash, digest,
    point_id, delete_chunk_ids, existing_manifest,
)

config = IngestorConfig()
print('Imported ingestion functions from ingestor.py')

c:\Users\ESHAAN\HAKUR\AI-GOAT\.AiGoatVenv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Imported ingestion functions from ingestor.py


## 1 — Structure-aware chunking (item 8)

Try splitting **at Markdown headings first**, then use your existing token-aware splitter within each section. This preserves heading context when the extracted text contains `# Heading` style markers.

**Limitation:** `PyPDFLoader` does not necessarily emit Markdown headings; plain PDFs may fall back to your original splitter. Later, a layout parser like Docling can supply better heading/table structure.

The output keeps your deterministic ID format, so it can be used with the current manifest/Qdrant scheme.

In [2]:
def structure_aware_chunk_source(path: Path, ingestor: Ingestor) -> list[Document]:
    path = Path(path).resolve()
    documents = standardized_docs(path, ingestor.config.DATA_DIR)
    heading_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[('#', 'h1'), ('##', 'h2'), ('###', 'h3')],
        strip_headers=False,
    )
    output = []
    for doc in documents:
        # Heading parser can lose source metadata; explicitly merge it back.
        sections = heading_splitter.split_text(doc.page_content)
        if not sections:
            sections = [Document(page_content=doc.page_content)]
        local_index = 0
        for section in sections:
            enriched = Document(
                page_content=section.page_content,
                metadata={**doc.metadata, **section.metadata},
            )
            for chunk in ingestor.splitter.split_documents([enriched]):
                if not chunk.page_content.strip():
                    continue
                key = (f"{doc.metadata['source_id']}:"
                       f"{doc.metadata['part_index']}:"
                       f"{local_index}:{chunk.page_content}")
                chunk.id = digest(key.encode('utf-8'))
                chunk.metadata['chunk_index'] = local_index
                output.append(chunk)
                local_index += 1
    return output

# Review without embeddings or database writes:
# ingestor = Ingestor(config)
# sample = structure_aware_chunk_source(ingestor.files[0], ingestor)
# [(c.metadata, c.page_content[:100]) for c in sample[:3]]

## 2 — Exact duplicates across new **and existing** chunks (item 10)

A hash of the normalized text identifies exact matches. Unlike the original notebook, these functions can compare new chunks to existing Qdrant payloads.

**Policy:** report duplicates; do not automatically delete them. Two sources can contain the same passage and both should remain citable. For a small local dataset, scrolling the collection is acceptable; for large datasets use a payload index/content-hash lookup rather than scanning everything.

In [3]:
def stored_chunks(ingestor: Ingestor):
    offset = None
    while True:
        points, offset = ingestor.client.scroll(
            collection_name=ingestor.config.COLLECTION,
            limit=256,
            offset=offset,
            with_payload=True,
            with_vectors=False,
        )
        for p in points:
            yield p
        if offset is None:
            break


def exact_duplicate_report(ingestor: Ingestor, incoming: list[Document]):
    grouped = defaultdict(list)
    for p in stored_chunks(ingestor):
        text = (p.payload or {}).get('text', '')
        if text:
            grouped[digest(text.encode('utf-8'))].append(
                ('qdrant', (p.payload or {}).get('source_path'), str(p.id))
            )
    for chunk in incoming:
        if chunk.page_content:
            grouped[digest(chunk.page_content.encode('utf-8'))].append(
                ('incoming', chunk.metadata.get('source_path'), chunk.id)
            )
    return {h: refs for h, refs in grouped.items() if len(refs) > 1}

# Read-only example (initialize ingestor first):
# candidate = structure_aware_chunk_source(ingestor.files[0], ingestor)
# matches = exact_duplicate_report(ingestor, candidate)
# print('Exact duplicate text groups:', len(matches))

## 3 — Near-duplicate candidate detection (item 11)

Use **RapidFuzz** to inspect similar chunks after removing exact matches. This catches small edits, but textual similarity is **not semantic equivalence**. A changed number or negation may be important.

The simple implementation below compares a **bounded sample** of incoming and existing chunks. It is a diagnostic, not a global deduplication engine. No source gets deleted.

In [4]:
def near_duplicate_report(ingestor: Ingestor, incoming: list[Document],
                          threshold: float = 90.0, max_existing: int = 200,
                          max_incoming: int = 50):
    existing = []
    for p in stored_chunks(ingestor):
        text = (p.payload or {}).get('text', '')
        if text:
            existing.append((str(p.id), text))
        if len(existing) >= max_existing:
            break
    hits = []
    for chunk in incoming[:max_incoming]:
        for old_id, old_text in existing:
            if chunk.page_content == old_text:
                continue  # exact duplicates have their own report
            score = ratio(chunk.page_content, old_text)
            if score >= threshold:
                hits.append((round(score, 1), chunk.id, old_id))
    return sorted(hits, reverse=True)

# print(near_duplicate_report(ingestor, candidate)[:10])

## 4 — Reuse stored embeddings for unchanged chunks (item 18)

**Core idea:** if a chunk ID is unchanged, its content and source/position key are unchanged; retrieve its vector from Qdrant and reuse it. Embed only IDs not already present. This avoids unnecessary Gemini calls for unchanged chunks of a modified file.

**Important:** this reuse is valid only within the **same collection and embedding model/configuration**. If you change the model or its output dimensions, use a new collection or re-embed. Our deterministic chunk IDs do not encode embedding-model version.

The function below is an **alternative to** `ingestor.ingest_one()`, not a monkey patch. It retains source-specific upsert/deletion behavior and updates SQLite after Qdrant operations. It is still not an atomic cross-database transaction.

In [5]:
def ingest_one_with_reuse(ingestor: Ingestor, path: Path,
                          use_structure: bool = True, batch_size: int = 64):
    path = Path(path).resolve()
    if not path.is_file() or path.suffix.lower() not in ingestor.config.SUPPORTED:
        raise ValueError(f'Missing or unsupported file: {path}')

    sid = source_id(path, ingestor.config.DATA_DIR)
    current_hash = file_hash(path)
    manifest = existing_manifest(ingestor.conn)
    previous = manifest.get(sid)
    if previous and previous['hash'] == current_hash:
        print('Unchanged; skipped:', source_key(path, ingestor.config.DATA_DIR))
        return {'skipped': True, 'reused': 0, 'embedded': 0}

    chunks = (structure_aware_chunk_source(path, ingestor) if use_structure
              else chunk_source(path, ingestor.config.DATA_DIR, ingestor.splitter))
    old_ids = set(previous['ids']) if previous else set()
    new_ids = {c.id for c in chunks}
    reused = embedded = 0

    for start in range(0, len(chunks), batch_size):
        batch = chunks[start:start + batch_size]
        # Look up existing points; ids may be the same even after an interrupted run.
        old_points = ingestor.client.retrieve(
            collection_name=ingestor.config.COLLECTION,
            ids=[point_id(c.id) for c in batch],
            with_vectors=True, with_payload=False,
        )
        by_id = {str(p.id): p.vector for p in old_points}
        missing = [c for c in batch if by_id.get(point_id(c.id)) is None]
        vectors_new = (ingestor.embeddings.embed_documents(
            [c.page_content for c in missing]
        ) if missing else [])
        if len(vectors_new) != len(missing):
            raise ValueError('Embedding count mismatch')
        generated = {point_id(c.id): vector for c, vector in zip(missing, vectors_new)}
        reused += len(batch) - len(missing)
        embedded += len(missing)
        points = [models.PointStruct(
            id=point_id(c.id),
            vector=(by_id.get(point_id(c.id)) if by_id.get(point_id(c.id)) is not None
                    else generated[point_id(c.id)]),
            payload={'text': c.page_content, **c.metadata, 'chunk_id': c.id},
        ) for c in batch]
        if points:
            ingestor.client.upsert(
                collection_name=ingestor.config.COLLECTION,
                points=points, wait=True,
            )

    delete_chunk_ids(list(old_ids - new_ids), ingestor.client, ingestor.config.COLLECTION)
    with ingestor.conn:
        ingestor.conn.execute(
            'INSERT OR REPLACE INTO sources VALUES (?, ?, ?, ?)',
            (sid, source_key(path, ingestor.config.DATA_DIR),
             current_hash, json.dumps([c.id for c in chunks])),
        )
    ingestor.manifest = existing_manifest(ingestor.conn)
    print(f'Indexed {len(chunks)} chunks; reused {reused}; embedded {embedded}')
    return {'skipped': False, 'reused': reused, 'embedded': embedded}

# Explicit opt-in: writes to Qdrant/SQLite and may invoke paid embeddings!
# ingestor = Ingestor(config)
# result = ingest_one_with_reuse(ingestor, Path('data/example.txt'))

## 5 — Validation and failure-recovery checks (item 19)

These are **read-only integrity checks**. We compare manifest chunk IDs with Qdrant point IDs, detect orphaned points, and verify that manifest paths still exist. This is much more informative than comparing only total point counts.

Run after ingesting, modifying, or deleting sample files. **If you see missing or orphaned IDs, investigate before deleting anything.** A crash may cause temporary disagreement between Qdrant and SQLite.

In [6]:
def validate_ingestion(ingestor: Ingestor):
    manifest = existing_manifest(ingestor.conn)
    manifest_points = {point_id(cid)
                       for entry in manifest.values()
                       for cid in entry['ids']}
    actual_points = {str(p.id) for p in stored_chunks(ingestor)}
    missing_paths = [entry['path'] for entry in manifest.values()
                     if not (ingestor.config.DATA_DIR / entry['path']).is_file()]
    result = {
        'manifest_sources': len(manifest),
        'manifest_chunks': len(manifest_points),
        'qdrant_points': len(actual_points),
        'missing_in_qdrant': sorted(manifest_points - actual_points),
        'orphaned_in_qdrant': sorted(actual_points - manifest_points),
        'missing_source_files': missing_paths,
    }
    for key, value in result.items():
        print(f'{key}: {len(value) if isinstance(value, list) else value}')
    return result

# report = validate_ingestion(ingestor)
# assert not report['missing_in_qdrant']
# assert not report['orphaned_in_qdrant']

## 6 — Tests without touching your real database

Use local, pure-function assertions to check the most important assumptions **without** creating a Qdrant collection or calling Gemini. Then separately test a disposable dataset/store with: first ingest, re-ingest unchanged, edit source, re-ingest, remove source, and interrupted run.

⚠️ To safely simulate crashes, use a **separate temporary Qdrant directory and SQLite manifest**, never your real `storage/` directory.

In [7]:
# No database writes or API usage:
assert digest(b'hello') == digest(b'hello')
assert digest(b'hello') != digest(b'Hello')
assert point_id('same-chunk') == point_id('same-chunk')
assert point_id('same-chunk') != point_id('another-chunk')
assert ratio('hello world', 'hello worlds') > 90
print('Pure-function checks passed')

Pure-function checks passed


## 7 — Manual integration checklist

- [ ] Same file ingested twice → second call skipped, zero embedding calls
- [ ] Edit one paragraph → unchanged chunk IDs reuse their stored vectors (when chunk boundaries remain stable)
- [ ] Duplicate text from two sources → report both source references, delete neither automatically
- [ ] Near-duplicate chunk → inspect similarity score; do **not** auto-delete
- [ ] Remove a file → call `ingestor.remove_source(sid)` explicitly, then validate
- [ ] Inject failure between Qdrant upsert and SQLite commit (disposable store) → rerun and validate
- [ ] Change embedding model/settings → re-embed or use a new collection

**Boundary:** These five sections provide implementations/experiments for items 8, 10, 11, 18 and 19. Near-duplicate handling is intentionally *review-only*, and Qdrant/SQLite consistency recovery is still retry-based rather than atomic.